# Heart Sound Classification — Baseline Workflow

This notebook contains the initial heart-sound classification workflow. Make sure the dataset and feature paths in the setup cells match your environment before running.

**Note:** Run cells in order and regenerate metrics in your own runtime; previous execution outputs have been cleared.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

print(os.listdir('/content/drive/MyDrive'))

In [ ]:
import os

FEATURE_DIR = "/content/drive/MyDrive/HeartSound_Project/fixed_log_mel_all"

files = [
    f for f in os.listdir(FEATURE_DIR)
    if f.endswith(".npy")
]

print("Feature files found:", len(files))
print("First 5:", files[:5])

In [ ]:
import pandas as pd
import os

BASE_DIR = "/content/drive/MyDrive/HeartSound_Project"

folders = [
    "training-a",
    "training-b",
    "training-c",
    "training-d",
    "training-e",
    "training-f"
]

reference_list = []

for folder in folders:
    ref_path = os.path.join(
        BASE_DIR,
        folder,
        "REFERENCE.csv"
    )

    df = pd.read_csv(
        ref_path,
        header=None,
        names=["recording", "label"]
    )

    df["folder"] = folder
    reference_list.append(df)

reference = pd.concat(
    reference_list,
    ignore_index=True
)

print("Total recordings:", len(reference))
print()
print(reference.head())
print()
print("Class distribution:")
print(reference["label"].value_counts())

In [ ]:
import os

dataset_df = reference.copy()

dataset_df["feature_path"] = dataset_df["recording"].apply(
    lambda x: os.path.join(
        FEATURE_DIR,
        x + ".npy"
    )
)

dataset_df["feature_exists"] = dataset_df["feature_path"].apply(
    os.path.exists
)

print("Total recordings:", len(dataset_df))
print("Features found:", dataset_df["feature_exists"].sum())
print("Features missing:", (~dataset_df["feature_exists"]).sum())

In [ ]:
from sklearn.model_selection import train_test_split

# Convert labels from -1/+1 to 0/1
dataset_df["target"] = dataset_df["label"].map({
    -1: 0,
     1: 1
})

# First split: 70% train, 30% temporary
train_df, temp_df = train_test_split(
    dataset_df,
    test_size=0.30,
    stratify=dataset_df["target"],
    random_state=42
)

# Second split: temporary → validation + test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["target"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

print("\nTrain labels:")
print(train_df["target"].value_counts())

print("\nValidation labels:")
print(val_df["target"].value_counts())

print("\nTest labels:")
print(test_df["target"].value_counts())

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class HeartSoundDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        # Load fixed Log-Mel feature
        feature = np.load(row["feature_path"])

        # Convert to float32 tensor
        feature = torch.tensor(
            feature,
            dtype=torch.float32
        )

        # Add channel dimension
        # (128, 954) → (1, 128, 954)
        feature = feature.unsqueeze(0)

        # Target
        target = torch.tensor(
            row["target"],
            dtype=torch.long
        )

        return feature, target


# Create datasets
train_dataset = HeartSoundDataset(train_df)
val_dataset = HeartSoundDataset(val_df)
test_dataset = HeartSoundDataset(test_df)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully.")

In [ ]:
features, targets = next(iter(train_loader))

print("Batch feature shape:", features.shape)
print("Batch target shape:", targets.shape)
print("Targets:", targets)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["target"].values
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float32
)

print("Training class counts:")
print(train_df["target"].value_counts())

print("\nClass weights:")
print(class_weights)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
import torch.nn as nn


class ImprovedHeartSoundCRNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.lstm = nn.LSTM(
            input_size=1024,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        # (B, 1, 128, 954)
        x = self.cnn(x)

        # (B, 64, 16, 119)
        batch_size = x.size(0)

        # (B, 119, 64, 16)
        x = x.permute(0, 3, 1, 2)

        # (B, 119, 1024)
        x = x.reshape(
            batch_size,
            119,
            1024
        )

        # BiLSTM
        x, _ = self.lstm(x)

        # Average over time
        x = x.mean(dim=1)

        # Classifier
        x = self.classifier(x)

        return x


crnn_model = ImprovedHeartSoundCRNN().to(device)

print(crnn_model)

In [ ]:
crnn_criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

crnn_optimizer = torch.optim.Adam(
    crnn_model.parameters(),
    lr=0.001
)

print("Device:", device)
print("Loss:", crnn_criterion)
print("Optimizer: Adam")

In [ ]:
features, targets = next(iter(train_loader))

features = features.to(device)
targets = targets.to(device)

with torch.no_grad():
    outputs = crnn_model(features)

print("Input:", features.shape)
print("Output:", outputs.shape)
print("Target:", targets.shape)
print("Device:", features.device)

In [ ]:
NUM_EPOCHS = 20

crnn_train_losses = []
crnn_val_losses = []
crnn_train_accuracies = []
crnn_val_accuracies = []

best_crnn_val_accuracy = 0.0
best_crnn_epoch = 0

CRNN_SAVE_PATH = (
    "/content/drive/MyDrive/HeartSound_Project/"
    "best_full_crnn.pth"
)

for epoch in range(NUM_EPOCHS):

    # =========================
    # TRAIN
    # =========================
    crnn_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        crnn_optimizer.zero_grad()

        outputs = crnn_model(features)

        loss = crnn_criterion(outputs, targets)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            crnn_model.parameters(),
            max_norm=1.0
        )

        crnn_optimizer.step()

        running_loss += loss.item() * features.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    train_loss = running_loss / total
    train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    crnn_model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            outputs = crnn_model(features)

            loss = crnn_criterion(outputs, targets)

            val_running_loss += loss.item() * features.size(0)

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    # =========================
    # SAVE HISTORY
    # =========================
    crnn_train_losses.append(train_loss)
    crnn_val_losses.append(val_loss)

    crnn_train_accuracies.append(train_accuracy)
    crnn_val_accuracies.append(val_accuracy)

    # =========================
    # SAVE BEST MODEL
    # =========================
    if val_accuracy > best_crnn_val_accuracy:

        best_crnn_val_accuracy = val_accuracy
        best_crnn_epoch = epoch + 1

        torch.save(
            crnn_model.state_dict(),
            CRNN_SAVE_PATH
        )

        print(
            f"  → Best CRNN saved "
            f"(Epoch {best_crnn_epoch}, "
            f"Val Acc: {best_crnn_val_accuracy:.4f})"
        )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_accuracy:.4f}"
    )

print("\nCRNN training complete.")
print("Best CRNN epoch:", best_crnn_epoch)
print(
    "Best CRNN validation accuracy:",
    best_crnn_val_accuracy
)
print("Model saved to:", CRNN_SAVE_PATH)

In [ ]:
import shutil
import os

SOURCE_DIR = "/content/drive/MyDrive/HeartSound_Project/fixed_log_mel_all"
LOCAL_FEATURE_DIR = "/content/fixed_log_mel_all"

if not os.path.exists(LOCAL_FEATURE_DIR):
    print("Copying features to Colab local storage...")
    shutil.copytree(SOURCE_DIR, LOCAL_FEATURE_DIR)
    print("Copy complete.")
else:
    print("Local feature folder already exists.")

print(
    "Local feature files:",
    len([f for f in os.listdir(LOCAL_FEATURE_DIR) if f.endswith(".npy")])
)

In [ ]:
FEATURE_DIR = "/content/fixed_log_mel_all"

In [ ]:
FEATURE_DIR = "/content/fixed_log_mel_all"

dataset_df["feature_path"] = dataset_df["recording"].apply(
    lambda x: os.path.join(FEATURE_DIR, x + ".npy")
)

dataset_df["feature_exists"] = dataset_df["feature_path"].apply(
    os.path.exists
)

print("Features found:", dataset_df["feature_exists"].sum())
print("Features missing:", (~dataset_df["feature_exists"]).sum())

In [ ]:
train_dataset = HeartSoundDataset(train_df)
val_dataset = HeartSoundDataset(val_df)
test_dataset = HeartSoundDataset(test_df)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders recreated using LOCAL storage.")

In [ ]:
import time

start = time.time()

features, targets = next(iter(train_loader))

elapsed = time.time() - start

print("Batch shape:", features.shape)
print("Time for first batch:", round(elapsed, 2), "seconds")

In [ ]:
# Recreate a fresh CRNN model
crnn_model = ImprovedHeartSoundCRNN().to(device)

crnn_criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

crnn_optimizer = torch.optim.Adam(
    crnn_model.parameters(),
    lr=0.001
)

print("Fresh CRNN created.")
print("Device:", device)

In [ ]:
CRNN_SAVE_PATH = (
    "/content/drive/MyDrive/HeartSound_Project/"
    "best_full_crnn.pth"
)

# Load the best saved CRNN
crnn_model.load_state_dict(
    torch.load(
        CRNN_SAVE_PATH,
        map_location=device
    )
)

crnn_model = crnn_model.to(device)
crnn_model.eval()

print("Best CRNN checkpoint loaded successfully.")
print("Best validation accuracy: 0.8683")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

all_predictions = []
all_targets = []

crnn_model.eval()

with torch.no_grad():

    for features, targets in test_loader:

        features = features.to(device)
        targets = targets.to(device)

        outputs = crnn_model(features)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            targets.cpu().numpy()
        )

# Metrics
accuracy = accuracy_score(
    all_targets,
    all_predictions
)

precision = precision_score(
    all_targets,
    all_predictions,
    zero_division=0
)

recall = recall_score(
    all_targets,
    all_predictions,
    zero_division=0
)

f1 = f1_score(
    all_targets,
    all_predictions,
    zero_division=0
)

cm = confusion_matrix(
    all_targets,
    all_predictions
)

# Confusion matrix:
# [[TN, FP],
#  [FN, TP]]

TN, FP, FN, TP = cm.ravel()

sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)

print("CRNN Test Results")
print("============================")
print(f"Accuracy:      {accuracy:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"F1-score:      {f1:.4f}")
print(f"Sensitivity:   {sensitivity:.4f}")
print(f"Specificity:   {specificity:.4f}")

print("\nConfusion Matrix:")
print(cm)